In [4]:
from backend.src.io import CDAweb

columns = ["tha_efs_dot0_gsm"]
instrument = "THA_L2_EFI"

data = CDAweb.default(instrument).get_dataset(columns, "2017-01-01T00:00:00", "2017-01-04T00:00:00")

2026-04-09 20:30:44.531 | INFO     | backend.src.io.cdaweb:get_dataset:20 - CDAWeb download dataset=THA_L2_EFI columns=['tha_efs_dot0_gsm'] since=2017-01-01T00:00:00 until=2017-01-04T00:00:00


In [5]:
data

<xarray.Dataset> Size: 3MB
Dimensions:                  (tha_efs_dot0_epoch: 93592,
                              tha_efs_dot0_gsm_compno: 3, record0: 155954)
Coordinates:
  * tha_efs_dot0_epoch       (tha_efs_dot0_epoch) datetime64[ns] 749kB 2017-0...
  * tha_efs_dot0_gsm_compno  (tha_efs_dot0_gsm_compno) int32 12B 1 2 3
    metavar0                 (tha_efs_dot0_gsm_compno) <U17 204B 'Ex GSM EFS_...
Dimensions without coordinates: record0
Data variables:
    tha_efs_dot0_epoch0      datetime64[ns] 8B 1970-01-01
    tha_efs_dot0_gsm         (tha_efs_dot0_epoch, tha_efs_dot0_gsm_compno) float32 1MB ...
    tha_efs_dot0_time        (record0) float64 1MB 1.483e+09 ... 1.484e+09
Attributes: (12/37)
    Project:                     ['THEMIS']
    Source_name:                 ['THA>Themis Probe A']
    Discipline:                  ['Space Physics>Magnetospheric Science']
    Data_type:                   ['EFI']
    Descriptor:                  ['L2>L2 DATA']
    Data_version:                ['1']
    ...                          ...
    PARENTS:                     ['tha_l2_efi_00000000_v01', 'tha_l2_efi_2016...
    Inst_settings:               ['Not used']
    Software_version:            ['8700']
    spase_DatasetResourceID:     ['spase://NASA/NumericalData/THEMIS/A/EFI/PT...
    DOI:                         ['https://doi.org/10.48322/3mhg-q627']
    CDAWEB_PARENTS:              ['tha_l2_efi_00000000_v01', 'tha_l2_efi_2016...

In [1]:
from backend.src.config import config

print(config.reading.delta)

1H


In [1]:
from backend.src.config import config
from backend.src.io import RawData

# Важно: это реальный запрос к CDAWeb и он может занять время.
# Для быстрой проверки используйте небольшой интервал в `backend/src/config/config.json`.

raw = RawData(config=config)

# Пример: скачивание FGM (магнитное поле) в DataFrame
df = raw.get_efi_dataframe()
# df.head()

EFI: скачивание пакетов:   0%|          | 0/4 [00:00<?, ?it/s]

2026-04-10 09:56:51.089 | INFO     | backend.src.io.cdaweb:get_dataset:20 - CDAWeb download dataset=THA_L2_EFI columns=['tha_efs_dot0_gsm'] since=2017-01-01T00:00:00Z until=2017-01-01T01:00:00Z
2026-04-10 09:56:54.549 | INFO     | backend.src.io.cdaweb:get_dataset:20 - CDAWeb download dataset=THA_L2_EFI columns=['tha_efs_dot0_gsm'] since=2017-01-01T01:00:00Z until=2017-01-01T02:00:00Z
2026-04-10 09:56:55.976 | INFO     | backend.src.io.cdaweb:get_dataset:20 - CDAWeb download dataset=THA_L2_EFI columns=['tha_efs_dot0_gsm'] since=2017-01-01T02:00:00Z until=2017-01-01T03:00:00Z
2026-04-10 09:56:57.439 | INFO     | backend.src.io.cdaweb:get_dataset:20 - CDAWeb download dataset=THA_L2_EFI columns=['tha_efs_dot0_gsm'] since=2017-01-01T03:00:00Z until=2017-01-01T04:00:00Z


## Черновые проверки: пути (`paths`) и `DataDownloading`

- **Без сети:** из `config` строятся сегмент `YYYY-MM-DD_YYYY-MM-DD`, папка `THEMIS-X` и полный путь к `.parquet`.
- **С CDAWeb:** внизу можно раскомментировать вызов `get_*_data(..., load=False)` — нужны `cdasws` и доступ в интернет.

In [3]:
# Примерные assert'ы для путей; сеть не нужна.
from backend.src.config import config
from backend.src.io.loader import DATASET_MAGNETIC_FIELD, DataDownloading
from backend.src.io.paths import (
    dataset_events_dir,
    event_interval_folder_name,
    parquet_dataset_path,
    themis_satellite_folder_name,
)


interval = event_interval_folder_name(config)
sat_dir = themis_satellite_folder_name(config)
events_dir = dataset_events_dir(config)
magnetic_path = parquet_dataset_path(config, DATASET_MAGNETIC_FIELD)



# --- опционально: реальное скачивание в backend/data/events/... (раскомментировать) ---
from backend.src.io.loader import DataDownloading
dl = DataDownloading(config)
df_efi = dl.get_efi_data(load=False)
print(df_efi.shape)
df_efi.head()

EFI: скачивание пакетов:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-10 12:09:11.197 | INFO     | backend.src.io.cdaweb:get_dataset:19 - CDAWeb download dataset=THA_L2_EFI columns=['tha_efs_dot0_gsm'] since=2017-01-01T00:00:00Z until=2017-01-04T00:00:00Z


(93592, 4)


,Time,GSM_Ex,GSM_Ey,GSM_Ez
0,2017-01-01 00:00:00.135,NaN,NaN,NaN
1,2017-01-01 00:00:02.896,NaN,NaN,NaN
2,2017-01-01 00:00:05.658,NaN,NaN,NaN
3,2017-01-01 00:00:08.419,NaN,NaN,NaN
4,2017-01-01 00:00:11.181,NaN,NaN,NaN
